In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import torch

from transformers import WhisperProcessor, WhisperForConditionalGeneration
from src.model import register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import processor_init, model_init, ModelArguments, LoraArguments, WhisperAccentTrainingArguments
from src.train.trainer import WhisperAccentTrainer
from peft import LoraConfig, get_peft_model
register_whisper_accent()

In [ ]:
import datetime

run_name = (
    f"whisper-accent-tiny-en-lora-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
)
output_dir = "/workspace/whisper-accent-tiny.en"
model_args = ModelArguments(
    model_type="whisper",
    base_model_name_or_path="openai/whisper-tiny.en",
    is_multilingual=False,
)
lora_args = LoraArguments()
training_args = WhisperAccentTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    lambda_accent_loss=0.1,
    lambda_diversity_loss=0.1,
    embedding_learning_rate=1e-4,
    learning_rate=1e-5,
    eval_on_start=True,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_steps=500,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=100,
    bf16=True,
    fp16=False,
    eval_steps=100,
    run_name=run_name,
    optim="adamw_torch",
    report_to=["tensorboard"],
    push_to_hub=True,
    logging_first_step=True,
    hub_model_id="mavleo96/whisper-accent-tiny.en",
    hub_strategy="all_checkpoints",
    gradient_checkpointing=False,
    predict_with_generate=True,
    remove_unused_columns=False,
)

In [ ]:
if model_args.model_type == "whisper":
    processor = WhisperProcessor.from_pretrained(model_args.base_model_name_or_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_args.base_model_name_or_path)
    if lora_args.lora_enable:
        target_modules = []
        for name, _ in model.named_modules():
            m_list = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
            if "model.decoder" in name and any(suffix in name for suffix in m_list):
                target_modules.append(name)
        lora_config = LoraConfig(
            r=lora_args.lora_r,
            lora_alpha=lora_args.lora_alpha,
            lora_dropout=lora_args.lora_dropout,
            bias=lora_args.lora_bias,
            use_rslora=lora_args.use_rslora,
            target_modules=target_modules,
            task_type=lora_args.task_type,
            ensure_weight_tying=True,
        )
        model = get_peft_model(model, lora_config)
    else:
        raise NotImplementedError("Non-LoRA training is not implemented yet")
elif model_args.model_type == "whisper_accent":
    processor = processor_init(model_args)
    model = model_init(model_args, lora_args, processor)

In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
)

train_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="train",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="validation",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)


In [ ]:
trainer = WhisperAccentTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    compute_metrics="all" if model_args.model_type == "whisper_accent" else "wer",
)


In [ ]:
trainer.train()
